<center><h1>TCGA-BRCA Dataset (Kaggle) Preprocessing</h1></center></br>

<h3>TCGA-BRCA Multi-Omics Dataset Overview</h3>

  <p> 
      <b>Sample Count:</b> 705 breast cancer samples</span> with complete data across all four omics modalities as provided in this 
      <a href="https://www.kaggle.com/datasets/samdemharter/brca-multiomics-tcga/data" target="_blank">Kaggle dataset</a>.
  </p>

  <div>
    <b>Omics Modalities & Feature Counts:</b>
    <ul>
      <li>cn (Copy Number Variations): 860 features</li>
      <li>mu (Mutations): 249 features</li>
      <li>rs (Gene Expression): 604 features</li>
      <li>pp (Protein Levels): 223 features</li>
    </ul>
    <p><b>Total Features: 1,936</b></p>
  </div>

  <P> <b>Data Form:</b> Preprocessed, gene/protein–level summaries (e.g., gene-level CNV calls, binary/numeric mutation indicators, normalized expression values, RPPA protein measurements) packaged in a single table for ready integration.
  </p>

  <div>
    <b>Acknowledgements:</b>
    <ul>
      <li>Original TCGA-BRCA data generated by The Cancer Genome Atlas (TCGA) project:
        <a href="https://portal.gdc.cancer.gov/projects/TCGA-BRCA" target="_blank">GDC Portal</a>.
      </li>
      <li>Aggregation and preprocessing by rbabaei in their analysis: 
        <a href="https://rbabaei82.github.io/MultiOmics_TCGA-BRCA/Analysis" target="_blank">MultiOmics_TCGA-BRCA Analysis</a>.
      </li>
      <li>Kaggle publication by Sam Demharter:
        <a href="https://www.kaggle.com/datasets/samdemharter/brca-multiomics-tcga/data" target="_blank">Dataset Page</a>.
      </li>
    </ul>
  </div>

  <div>
    <b>Sample Coverage: </b>
    <ul>
      <li><b>GDC Data Portal:</b> The TCGA-BRCA cohort on GDC includes around 1,094-1,098 primary tumor samples for modalities like CNV and expression, with clinical and various data categories available for ~1,096 patients.
      </li>
      <li><b>Kaggle Dataset:</b> Contains only 705 samples pecifically those with complete data across all four omics types (CNV, mutation, expression, protein). This subset ensures no missing modality for integrative analyses but reduces the total sample size relative to the full GDC cohort.
      </li>
    </ul>
  </div>

</body>
</html>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
original_tcga_brca_dataset = pd.read_csv('TCGA-BRCA.csv')
tcga_brca_data = original_tcga_brca_dataset.copy()

print('Original TCGA-BRCA Dataset Info: ')
tcga_brca_data.info()

tcga_brca_data.columns

Original TCGA-BRCA Dataset Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 705 entries, 0 to 704
Columns: 1941 entries, rs_CLEC3A to histological.type
dtypes: float64(827), int64(1110), object(4)
memory usage: 10.4+ MB


Index(['rs_CLEC3A', 'rs_CPB1', 'rs_SCGB2A2', 'rs_SCGB1D2', 'rs_TFF1',
       'rs_MUCL1', 'rs_GSTM1', 'rs_PIP', 'rs_ADIPOQ', 'rs_ADH1B',
       ...
       'pp_p62.LCK.ligand', 'pp_p70S6K', 'pp_p70S6K.pT389', 'pp_p90RSK',
       'pp_p90RSK.pT359.S363', 'vital.status', 'PR.Status', 'ER.Status',
       'HER2.Final.Status', 'histological.type'],
      dtype='object', length=1941)

In [3]:
# Check for missing values
print("\nNo of Missing Values-Feature Dataset:", tcga_brca_data.isnull().sum().sum())
print("\n Missing values of Feature Dataset:\n", tcga_brca_data.isnull().sum())


No of Missing Values-Feature Dataset: 389

 Missing values of Feature Dataset:
 rs_CLEC3A              0
rs_CPB1                0
rs_SCGB2A2             0
rs_SCGB1D2             0
rs_TFF1                0
                    ... 
vital.status           0
PR.Status            122
ER.Status            122
HER2.Final.Status    145
histological.type      0
Length: 1941, dtype: int64


In [4]:
# Handle missing values
subset = ["ER.Status", "PR.Status", "HER2.Final.Status"]

tcga_brca_data = tcga_brca_data.dropna(subset=subset)

print("\nNo of Missing Values - Feature Dataset:", tcga_brca_data.isnull().sum().sum(), "\n")
tcga_brca_data.info()


No of Missing Values - Feature Dataset: 0 

<class 'pandas.core.frame.DataFrame'>
Index: 560 entries, 0 to 649
Columns: 1941 entries, rs_CLEC3A to histological.type
dtypes: float64(827), int64(1110), object(4)
memory usage: 8.3+ MB


In [5]:
for col in subset:
    print(f'Unique values in {col}:')
    print(tcga_brca_data[col].unique(), "\n")

for col in subset:
    uncommon_values = tcga_brca_data[col].unique().tolist()
    count = tcga_brca_data[col].isin(
        set(uncommon_values).difference(['Positive', 'Negative'])
    ).sum()
    print(f'Uncommon values (not Positive/Negative) in {col}: {count}')

Unique values in ER.Status:
['Positive' 'Negative' 'Performed but Not Available' 'Indeterminate'
 'Not Performed'] 

Unique values in PR.Status:
['Positive' 'Negative' 'Performed but Not Available' 'Indeterminate'
 'Not Performed'] 

Unique values in HER2.Final.Status:
['Negative' 'Positive' 'Equivocal' 'Not Available'] 

Uncommon values (not Positive/Negative) in ER.Status: 31
Uncommon values (not Positive/Negative) in PR.Status: 34
Uncommon values (not Positive/Negative) in HER2.Final.Status: 17


In [1]:
tcga_brca_data = tcga_brca_data[
    (tcga_brca_data["HER2.Final.Status"].isin(["Positive", "Negative"])) &
    (tcga_brca_data["ER.Status"].isin(["Positive", "Negative"]))
]

tcga_brca_data.loc[
    ~tcga_brca_data["PR.Status"].isin(["Positive", "Negative"]),
    "PR.Status"
] = "Unknown"

tcga_brca_data.info()

NameError: name 'tcga_brca_data' is not defined

In [1]:
# Encode categorical and binary features

from sklearn.preprocessing import LabelEncoder

binary_columns = ['ER.Status', 'HER2.Final.Status']

for b_col in binary_columns:
    tcga_brca_data[b_col] = tcga_brca_data[b_col].apply(lambda x: 1 if str(x).lower() == 'positive' else 0)

label_histological_type = LabelEncoder()
label_pr_status = LabelEncoder()

tcga_brca_data['histological.type'] = label_histological_type.fit_transform(tcga_brca_data['histological.type'])
tcga_brca_data['PR.Status'] = label_pr_status.fit_transform(tcga_brca_data['PR.Status'])

# Display the mapping
histological_type_map = dict(zip(range(len(label_histological_type.classes_)), label_histological_type.classes_))
print("\nEncoded Values to Histological Type Mapping:\n")
for enco_val, histological_type in histological_type_map.items():
    print(enco_val," --> ", histological_type)

pr_status_map = dict(zip(range(len(label_pr_status.classes_)), label_pr_status.classes_))
print("\nEncoded Values to PR Status Mapping:\n")
for enco_val, pr_status in pr_status_map.items():
    print(enco_val," --> ", pr_status)

NameError: name 'tcga_brca_data' is not defined

In [8]:
multi_omics = set()
status = set()
subtypes = {'Luminal-A', 'Luminal-B', 'HER2-Enriched', 'Basal-Like', 'Undefined'}

for col in tcga_brca_data.columns:
    if '_' in col:
        multi_omics.add(col.split('_')[0])
    else:  
        status.add(col)

print("\nMulti-Omics: ", multi_omics,'\nSubtypes: ', subtypes,'\nStatus: ', status)


Multi-Omics:  {'pp', 'rs', 'mu', 'cn'} 
Subtypes:  {'Luminal-B', 'Undefined', 'Luminal-A', 'Basal-Like', 'HER2-Enriched'} 
Status:  {'PR.Status', 'histological.type', 'HER2.Final.Status', 'ER.Status', 'vital.status'}


In [9]:
for subtype in subtypes:
    tcga_brca_data[subtype] = 0

for index, row in tcga_brca_data.iterrows():
    er_status = row['ER.Status']
    pr_status = row['PR.Status']
    her2_status = row['HER2.Final.Status']

    if er_status == 1 and pr_status == 1 and her2_status == 0:
        tcga_brca_data.at[index, 'Luminal-A'] = 1

    elif er_status == 1 and pr_status in [2, 1, 0] and her2_status == 1:
        tcga_brca_data.at[index, 'Luminal-B'] = 1

    elif er_status == 0 and pr_status == 0 and her2_status == 1:
        tcga_brca_data.at[index, 'HER2-Enriched'] = 1

    elif er_status == 0 and pr_status == 0 and her2_status == 0:
        tcga_brca_data.at[index, 'Basal-Like'] = 1

    else : tcga_brca_data.at[index, 'Undefined'] = 1

print('\nSubtype Labled TCGA-BRCA Dataset Info: \n')
tcga_brca_data.info()


Subtype Labled TCGA-BRCA Dataset Info: 

<class 'pandas.core.frame.DataFrame'>
Index: 512 entries, 0 to 649
Columns: 1946 entries, rs_CLEC3A to HER2-Enriched
dtypes: float64(827), int32(2), int64(1117)
memory usage: 7.6 MB


In [10]:
columns_of_each_multi_omics = {}

for omic_type in multi_omics:
    columns_of_each_multi_omics[omic_type] = []
    for col in tcga_brca_data.columns:
        if (omic_type + '_') in col:
            columns_of_each_multi_omics[omic_type].append(col)

In [11]:
gene_data = tcga_brca_data[columns_of_each_multi_omics['rs']]
protein_data = tcga_brca_data[columns_of_each_multi_omics['pp']]
cnv_data = tcga_brca_data[columns_of_each_multi_omics['cn']]
mutation_data = tcga_brca_data[columns_of_each_multi_omics['mu']]

status_data = tcga_brca_data[list(status)]
subtype_data = tcga_brca_data[list(subtypes)]

print('\nGene Records Info:')
gene_data.info()

print('\nProtein Records Info:')
protein_data.info()

print('\nCopy Number Variation(CNV) Records Info:')
cnv_data.info()

print('\nMutation Records Info:')
mutation_data.info()

subtype_counts = subtype_data.sum()
print("\nSample count per subtype:\n", subtype_counts, "\n")


Gene Records Info:
<class 'pandas.core.frame.DataFrame'>
Index: 512 entries, 0 to 649
Columns: 604 entries, rs_CLEC3A to rs_HEPN1
dtypes: float64(604)
memory usage: 2.4 MB

Protein Records Info:
<class 'pandas.core.frame.DataFrame'>
Index: 512 entries, 0 to 649
Columns: 223 entries, pp_X14.3.3.beta to pp_p90RSK.pT359.S363
dtypes: float64(223)
memory usage: 912.2 KB

Copy Number Variation(CNV) Records Info:
<class 'pandas.core.frame.DataFrame'>
Index: 512 entries, 0 to 649
Columns: 860 entries, cn_ISG15 to cn_MLC1
dtypes: int64(860)
memory usage: 3.4 MB

Mutation Records Info:
<class 'pandas.core.frame.DataFrame'>
Index: 512 entries, 0 to 649
Columns: 249 entries, mu_ANK3 to mu_PEG3
dtypes: int64(249)
memory usage: 1016.2 KB

Sample count per subtype:
 Luminal-B         56
Undefined         62
Luminal-A        279
Basal-Like        90
HER2-Enriched     25
dtype: int64 



In [12]:
def getPreprocessedData():
    data_dict = {
        "gene_data": gene_data,
        "protein_data": protein_data,
        "cnv_data": cnv_data,
        "mutation_data": mutation_data,
        "status_data": status_data,
        "subtype_data": subtype_data,
        "tcga_brca_data": tcga_brca_data 
    }
    return data_dict